[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/rewrite/rewrite/examples/mechanics/stiffness.ipynb) [![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/EelcoHoogendoorn/numga?ref=rewrite)

# Planar Rigid-Body Stiffness, Inertia & Vibration Modes in PGA2D

When an elastic spring suspension supports a rigid body, its natural vibration frequencies and mode shapes depend on the interplay between elastic stiffness and mass inertia.

Rather than assembling coordinate mass and stiffness matrices by hand, this notebook uses **extensors** (linear maps between blade subspaces). In PGA, **both stiffness and inertia are purely additive quantities**: rank-1 spring lines compile directly into a **stiffness extensor**, mass points compile into an **inertia extensor** without origin choices or parallel-axis theorems, and normal vibration modes emerge from a single generalized eigensolve on their bilinear energy forms.


In [ ]:
# Setup environment: clone repository and configure paths if running in Colab
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    import os, shutil, subprocess
    os.chdir("/content")
    repo_dir = Path("/content/numga_repo")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", "-b", "rewrite", "https://github.com/EelcoHoogendoorn/numga.git", str(repo_dir)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo_dir), "pull", "origin", "rewrite"], check=False)
    rewrite_dir = repo_dir / "rewrite"
    src_dir = rewrite_dir / "src"
    for p in [str(src_dir), str(rewrite_dir)]:
        if p not in sys.path:
            sys.path.insert(0, p)

# Ensure rewrite root is in sys.path when running locally:
for cand in [Path.cwd(), *Path.cwd().parents]:
    if (cand / "examples" / "mechanics").is_dir():
        if str(cand) not in sys.path:
            sys.path.insert(0, str(cand))
        break
    if (cand / "rewrite" / "examples" / "mechanics").is_dir():
        p = str(cand / "rewrite")
        if p not in sys.path:
            sys.path.insert(0, p)
        break


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image
from pathlib import Path

from numga import NumpyContext
from numga.algebras import PGA2D
from examples.mechanics.stiffness_plumbing import (
    render_setup,
    render_mass_distribution,
    render_modes,
    render_animation,
)

# Bind 2D Projective Geometric Algebra (PGA2D):
ctx = NumpyContext(PGA2D)
mv = ctx.multivector

# Blade Subspaces:
Scalar = PGA2D.gatype.scalar()              # Grade 0: scalar real values
Point = PGA2D.gatype.antivector()           # Grade 2: projective points in the 2D plane
Twist = PGA2D.gatype.bivector()             # Grade 2 dual / Lie algebra: infinitesimal rigid motions & velocities
Wrench = PGA2D.gatype.vector()              # Grade 1: lines of action for forces, torques, and springs

# Extensors (linear maps between blade subspaces):
SpringExtension = PGA2D.gatype((Scalar, Twist))   # Scalar <- Twist: measures spring extension from body motion
Stiffness = PGA2D.gatype((Wrench, Twist))         # Wrench <- Twist: restoring force and torque from body motion
Inertia = PGA2D.gatype((Wrench, Twist))           # Wrench <- Twist: kinetic momentum wrench from twist velocity

def point(xy: np.ndarray) -> Point:
    """Embed (..., 2) Cartesian positions as unit-weight PGA points."""
    xy = np.asarray(xy, dtype=float)
    return mv.antivector(np.concatenate([xy, np.ones_like(xy[..., :1])], axis=-1))

print("PGA2D Algebra and Extensor types initialized successfully.")


## 1. Rigid Body Geometry & Springs

A uniform 2x1 rectangular plate of mass 1 is suspended in the plane by two vertical elastic springs.

In PGA, a spring is the line joining its anchor to its attachment point: `lines = (anchors & attachments).normalized()`. Below, we set up the plate vertices, the two vertical springs, and plot the baseline suspension at rest.


In [ ]:
# 1. Uniform 2x1 rigid plate vertices:
body = point([[-1.0, -0.5], [1.0, -0.5], [1.0, 0.5], [-1.0, 0.5]])

# 2. Baseline suspension: two vertical springs (anchors directly above attachments):
attachments = point([[-0.8, 0.5], [0.8, 0.5]])
anchors = point([[-0.8, 1.55], [0.8, 1.55]])
k = mv.scalar(np.full((2, 1), 6.0))

# 3. Spring lines of action in PGA (joining anchor to attachment):
lines = (anchors & attachments).normalized()   # [2] Wrench

# Plot 1: Setup - Rigid plate suspended by vertical springs at equilibrium
fig = render_setup(
    body=body,
    anchors_list=anchors,
    attachments_list=attachments,
    titles="Baseline Suspension: Two Vertical Springs",
    plot_path="examples/plots/stiffness_setup.png",
)
plt.show()


## 2. Building the Stiffness Extensor from Springs

In classical mechanics, assembling a rigid-body stiffness matrix requires choosing an origin, computing moment arms, and projecting force vectors.

In PGA, stiffness is a purely additive quantity. Each spring defines an independent rank-1 extensor (a dyad mapping rigid motion to a restoring line of force: `lines * (Twist & lines) * k`). Summing these individual spring extensors directly yields the total stiffness of the suspension, naturally combining linear resistance and restoring torques without choosing a reference origin.


In [ ]:
# 1. Measure spring extension from an open rigid-body motion (linear form: Twist -> Scalar):
extension = Twist & lines

# 2. Hooke's law: line of action scaled by extension and spring constant (Wrench <- Twist):
spring_stiffness = lines * extension * k

# 3. Stiffness is purely additive: summing spring extensors yields total suspension stiffness:
stiffness = spring_stiffness.sum(axis=0)

print("Stiffness extensor GAType:", stiffness.gatype)
print("Kernel shape             :", stiffness.kernel.shape, " # 3x3 extensor: Wrench <- Twist")
print("Stiffness matrix (Two vertical springs):\n", np.round(stiffness.kernel, 3))


## 3. Building the Inertia Extensor from Mass Points

In classical mechanics, rigid-body inertia is split between total mass, center of mass, and a rotational inertia matrix, combined using parallel-axis (Huygens-Steiner) theorems.

In PGA, rigid-body inertia mirrors stiffness as a purely additive extensor mapping twist velocity to momentum wrench (`Wrench <- Twist`). The commutator of a mass point with an open twist (`mass_points.commutator(Twist)`) gives its direction of motion. Joining the point with its velocity direction (`mass_points & velocities`) forms the line of action of its linear momentum, which mass scales into a momentum wrench.

Summing these point extensors automatically synthesizes total mass, center of mass, and rotational inertia without origin shifts or parallel-axis formulas. Below, 4 Gauss quadrature points capture the 2x1 plate's mass (1.0 kg) and polar moment of inertia (5/12 kg*m^2) exactly.


In [ ]:
# 1. Mass distribution via 2-point Gauss quadrature on the 2x1 rectangle:
mass_coords = np.array([[-1.0, -0.5], [1.0, -0.5], [1.0, 0.5], [-1.0, 0.5]]) / np.sqrt(3)
mass_points = point(mass_coords)
masses = mv.scalar(np.full((4, 1), 0.25))

# 2. Direction of motion (velocity) of each point under an open twist (Lie bracket):
velocities = mass_points.commutator(Twist)

# 3. Join each point with its direction of motion to form the momentum line, scaled by mass:
point_momenta = (mass_points & velocities) * masses

# 4. Inertia is purely additive: summing point extensors automatically yields total inertia:
inertia = point_momenta.sum(axis=0)

# Plot 2: Mass distribution - 4-point Gauss quadrature on the rigid plate
fig = render_mass_distribution(
    body=body,
    mass_points=mass_points,
    plot_path="examples/plots/stiffness_mass.png",
)
plt.show()

print("Inertia extensor GAType:", inertia.gatype)
print("Kernel shape           :", inertia.kernel.shape, " # 3x3 extensor: Wrench <- Twist")
print("Inertia matrix (mass = 1.0, polar moment = 5/12 ≈ 0.417):\n", np.round(inertia.kernel, 3))


## 4. Modal Analysis of the Baseline Suspension

Natural vibration modes solve the generalized eigenvalue problem between elastic potential energy and kinetic energy: `(Twist & stiffness).eigh(Twist & inertia)`.

The eigenvalues yield the natural frequencies, while eigenvectors give the normal mode twists. Because the two vertical springs are symmetric, sideways motion leaves both springs unchanged to first order, producing a 0 Hz free slide mechanism alongside decoupled vertical bounce and rocking modes. Below, we plot the three independent mode shapes.


In [ ]:
# 1. Solve generalized eigenvalue problem directly on bilinear energy forms:
values, modes = (Twist & stiffness).eigh(Twist & inertia)

# 2. Natural frequencies in Hz: f = sqrt(lambda) / (2 * pi)
freqs = np.sqrt(np.maximum(values.kernel[..., 0], 0.0)) / (2 * np.pi)

print("=== Baseline Suspension: Two Vertical Springs ===")
for label, freq in zip(["Free slide", "Bounce", "Rock"], freqs):
    detail = "0.00 Hz (free mechanism)" if freq == 0 else f"{freq:.3f} Hz"
    print(f"  {label:<16}: {detail}")

# Plot 3: 3-Panel Baseline Mode Shapes (Uncoupled Motions)
fig = render_modes(
    body=body,
    anchors_list=anchors,
    attachments_list=attachments,
    modes_list=modes,
    values_list=values,
    extensions_list=extension,
    titles="Baseline Suspension: Two Vertical Springs (Uncoupled Modes)",
    descriptions="Sideways motion leaves both vertical springs unchanged to first order",
    labels_list=("Free slide", "Bounce", "Rock"),
    plot_path="examples/plots/stiffness_baseline.png",
)
plt.show()


## 5. Adding an Angled Spring: Additivity in Action

Adding an off-center angled spring breaks symmetry and constrains the free slide.

Because stiffness is purely additive in PGA, we do not recompute from scratch or project forces along new coordinate axes. We simply compute the rank-1 extensor of the new spring and add it directly: `stiffness_coupled = stiffness + line_extra * (Twist & line_extra) * k_extra`.

Solving the new generalized eigenvalue problem shows that the 0 Hz mechanism is eliminated, coupling horizontal translation, vertical bounce, and rocking into three distinct non-zero natural frequencies. Below, we compare the baseline and coupled mode shapes side-by-side.


In [ ]:
# 1. Define the third off-center angled spring:
att_extra = point([1.0, 0.0])
anc_extra = point([1.9, 0.85])
k_extra = mv.scalar([6.0])

line_extra = (anc_extra & att_extra).normalized()
ext_extra = Twist & line_extra
spring_extra = line_extra * ext_extra * k_extra

# 2. Additivity in action: add the new spring extensor directly to the existing stiffness:
stiffness_coupled = stiffness + spring_extra

# 3. Solve the generalized eigenvalue problem for the coupled system:
values_coupled, modes_coupled = (Twist & stiffness_coupled).eigh(Twist & inertia)
freqs_coupled = np.sqrt(np.maximum(values_coupled.kernel[..., 0], 0.0)) / (2 * np.pi)

print("=== Coupled Suspension: Added Angled Spring ===")
for label, freq in zip(["Coupled mode 1", "Coupled mode 2", "Coupled mode 3"], freqs_coupled):
    print(f"  {label:<16}: {freq:.3f} Hz")

# Combine spring geometry for the 3-spring case:
att_coupled = point([[-0.8, 0.5], [0.8, 0.5], [1.0, 0.0]])
anc_coupled = point([[-0.8, 1.55], [0.8, 1.55], [1.9, 0.85]])
ext_coupled = Twist & (anc_coupled & att_coupled).normalized()

# Plot 4: 6-Panel Mode Comparison - Baseline vs Coupled
fig = render_modes(
    body=body,
    anchors_list=[anchors, anc_coupled],
    attachments_list=[attachments, att_coupled],
    modes_list=[modes, modes_coupled],
    values_list=[values, values_coupled],
    extensions_list=[extension, ext_coupled],
    titles=("Baseline Suspension: Two Vertical Springs", "Coupled Suspension: Added Angled Spring"),
    plot_path="examples/plots/stiffness.png",
)
plt.show()


## 6. Harmonic Oscillation Animation

Releasing the plate from rest in each mode demonstrates physical time evolution.

Zero-frequency modes remain statically displaced, while non-zero modes oscillate at their computed natural frequencies. All panels share synchronized physical time.


In [ ]:
# Plot 5: Synchronized Harmonic Vibration Animation
gif_path = Path("examples/plots/stiffness.gif")
render_animation(
    body=body,
    anchors_list=[anchors, anc_coupled],
    attachments_list=[attachments, att_coupled],
    modes_list=[modes, modes_coupled],
    values_list=[values, values_coupled],
    extensions_list=[extension, ext_coupled],
    titles=("Baseline Suspension: Two Vertical Springs", "Coupled Suspension: Added Angled Spring"),
    animation_path=str(gif_path),
)

if gif_path.exists():
    display(Image(filename=str(gif_path)))


## 7. Summary & Conceptual Synthesis

Having walked through the working pipeline, we can step back and examine how **extensors** structure rigid-body mechanics:

* **Measuring Deformation with Lines (Spring Extension as a Linear Form)**:
  A spring is a line joining an anchor to an attachment point (`lines = anchors & attachments`). Pairing that line with an open twist (`Twist & lines`) directly yields a linear form (`Twist -> Scalar`) measuring spring stretch under any rigid body motion without coordinate projection formulas.

* **Additivity of Stiffness (Rank-1 Extensor Dyads)**:
  Each spring defines an independent rank-1 extensor dyad (`lines * (Twist & lines) * k`) mapping displacement twist to restoring wrench (`Wrench <- Twist`). Total stiffness is purely additive: summing individual spring extensors seamlessly combines linear forces and torques without choosing a reference origin. Adding a spring is literally adding its extensor.

* **Additivity of Inertia (Joining Points with Motion Lines)**:
  The commutator of a mass point with an open twist (`mass_points.commutator(Twist)`) gives its direction of motion. Joining the point with its velocity direction (`mass_points & velocities`) forms the line of action of its linear momentum (`Wrench <- Twist`). Summing across mass points synthesizes total mass, center of mass, and rotational inertia into one coordinate-free extensor without parallel-axis (Steiner) shifts.

* **Solving Modes Directly on Bilinear Energy Forms (Generalized Eigensolve)**:
  Rather than building coordinate mass and stiffness matrices, the vibration eigenvalue problem is solved directly between the potential energy form and kinetic energy form: `(Twist & stiffness).eigh(Twist & inertia)`.

* **Evaluating Physical Displacements (Lie Algebra Commutators)**:
  The motion of any point on the rigid body under an eigenvector twist step is evaluated directly via the Lie algebra commutator (`body.commutator(modes)`), cleanly connecting abstract Lie algebra coordinates to physical geometry.
